In [0]:
%python
import pyspark.sql.functions as F
df = spark.read.table("bronze_workforce.azure_blob_storage.employee")
display(df)

In [0]:
%python
df=df.drop('_file','_line','_modified','_fivetran_synced')
display(df)

In [0]:
%python
df = df.dropDuplicates()
display(df)

In [0]:
df = df.withColumn('full_name', F.concat(df.first_name, F.lit(' '), df.last_name))
df=df.drop('first_name','last_name')
display(df)

In [0]:
df = df.withColumn("employee_id",df["employee_id"].cast('int'))
df = df.withColumn("company_id",df["company_id"].cast('int'))
df = df.withColumn("department_id",df["department_id"].cast('int'))
display(df)

In [0]:
#df=F.to_date(df["hire_date"], format='yyyy-MM-dd')
df = df.withColumn('hire_date',F.trim(F.col("hire_date")))
df = df.withColumn("hire_date", F.to_timestamp(F.col("hire_date"), "dd-MM-yyyy HH:mm"))
# df = df.withColumn("hire_date", F.to_date(F.col("hire_date"), "dd-MM-yyyy HH:mm"))
display(df)


In [0]:
df = df.withColumn("hire_date", F.to_date(F.col("hire_date"), "dd-MM-yyyy"))
display(df)

In [0]:
df = df.withColumn('termination_date',F.trim(F.col("termination_date")))
df = df.withColumn("termination_date", F.to_timestamp(F.col("termination_date"), "dd-MM-yyyy HH:mm"))
df = df.withColumn("termination_date", F.to_date(F.col("termination_date"), "dd-MM-yyyy"))
display(df)

In [0]:
cols = df.columns
emplyee_code_idx = cols.index('employee_code')
new_cols = cols[:emplyee_code_idx+1] + ['full_name']
new_cols+= [col for col in cols if col != 'full_name' and col not in new_cols]
df = df.select(new_cols)
display(df)

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("silver_workforce.azure_blob_storage.employee")
